# Pose experiment suite — real / synthetic / mixed

Repete a avaliação de `classifiers_robot_emotions.ipynb` para três configurações de fonte de pose:

| Experimento | Pose | IMU sintético |
|---|---|---|
| `real_pose` | MotionBERT (vídeo real) | pipeline real |
| `synthetic_pose` | Kimodo (geração a partir de janelas reais) | pipeline Kimodo |
| `mixed_pose` | real + sintético juntos | idem ao modo correspondente |

Cada experimento de pose roda exatamente o mesmo `run_experiment_suite` do notebook original.

In [1]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / 'pose_module').exists() and (root / 'evaluation').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.classifiers import (
    ALL_CAPTURE_BLACKLIST,
    WindowedDatasetConfig,
    build_classifier_capture_table,
    build_windowed_multimodal_dataset,
)

try:
    from evaluation.classifiers import (
        EXPERIMENT_SPECS,
        ModelConfig,
        SplitConfig,
        TrainingConfig,
        run_experiment_suite,
    )
    TORCH_READY = True
    TORCH_IMPORT_ERROR = None
except ImportError as exc:
    TORCH_READY = False
    TORCH_IMPORT_ERROR = exc

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 260)
PROJECT_ROOT

PosixPath('/home/henriquesouza/IMUGPT')

## Configuração dos experimentos

- `REAL_OUTPUT_ROOT` — diretório do pipeline real (contém `virtual_imu_manifest.jsonl` + `imu.npz` por clipe)
- `SYNTHETIC_MANIFEST` — manifesto gerado por `export-kimodo-virtual-imu`
- `MIXED_MANIFEST` — manifesto gerado por `export-mixed-virtual-imu`

In [2]:
REAL_OUTPUT_ROOT     = PROJECT_ROOT / 'output' / 'exp_real_pose'
SYNTHETIC_MANIFEST   = PROJECT_ROOT / 'output' / 'exp_synthetic_pose' / 'virtual_imu_manifest.jsonl'
MIXED_MANIFEST       = PROJECT_ROOT / 'output' / 'exp_mixed_pose'      / 'mixed_virtual_imu_manifest.jsonl'
ANCHOR_CATALOG       = PROJECT_ROOT / 'output' / 'robot_emotions_kimodo_anchors_effectors_5' / 'kimodo_anchor_catalog.jsonl'

IMU_FEATURE_MODE = 'acc_euler'

# Emoções/estímulos excluídos manualmente da classificação (amostras insuficientes por sujeito)
EXCLUDED_EMOTIONS = {'Fear', 'Anger'}
EXCLUDED_STIMULI  = {'Simulation'}  # exclusivo de Fear/Anger

DATASET_CONFIG = WindowedDatasetConfig(
    window_size=81,
    overlap=0.5,
    synthetic_variant='raw',
    imu_feature_mode=IMU_FEATURE_MODE,
    selected_sensors=None,
    max_windows_per_capture=None,
    random_state=42,
)

SPLIT_CONFIG = SplitConfig(n_splits=5, random_state=42) if TORCH_READY else None
MODEL_CONFIG = ModelConfig(hidden_dim=128, dropout=0.1, trunk_blocks=2, modality_dropout_p=0.1) if TORCH_READY else None
TRAINING_CONFIG = TrainingConfig(
    batch_size=64, max_epochs=10, learning_rate=2e-3, weight_decay=1e-4,
    device='cuda',
    domain_loss_weight=0.1, flat_tag_loss_weight=0.3, emotion_loss_weight=1.0,
    modality_loss_weight=0.3, stimulus_loss_weight=0.3, use_cb_focal=False,
) if TORCH_READY else None

REQUESTED_ORDER = [
    'vision_only', 'imu_only_r2r', 'imu_only_s2r', 'imu_only_mixed2r',
    'vision_imu_r2r', 'vision_imu_s2r', 'vision_imu_mixed2r',
]

print('REAL_OUTPUT_ROOT:', REAL_OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST)
print('MIXED_MANIFEST:', MIXED_MANIFEST)
print('ANCHOR_CATALOG:', ANCHOR_CATALOG)
print('TORCH_READY:', TORCH_READY)
print('EXCLUDED_EMOTIONS:', EXCLUDED_EMOTIONS)
print('EXCLUDED_STIMULI:', EXCLUDED_STIMULI)

REAL_OUTPUT_ROOT: /home/henriquesouza/IMUGPT/output/exp_real_pose
SYNTHETIC_MANIFEST: /home/henriquesouza/IMUGPT/output/exp_synthetic_pose/virtual_imu_manifest.jsonl
MIXED_MANIFEST: /home/henriquesouza/IMUGPT/output/exp_mixed_pose/mixed_virtual_imu_manifest.jsonl
ANCHOR_CATALOG: /home/henriquesouza/IMUGPT/output/robot_emotions_kimodo_anchors_effectors_5/kimodo_anchor_catalog.jsonl
TORCH_READY: True
EXCLUDED_EMOTIONS: {'Anger', 'Fear'}
EXCLUDED_STIMULI: {'Simulation'}


## Helpers para carregar manifestos sintéticos

O manifesto sintético não tem `domain`/`user_id`/`tag_number` — esses campos vêm do manifesto real
via `reference_clip_id`. A função abaixo constrói uma capture table compatível com
`build_windowed_multimodal_dataset`.

In [3]:
from pose_module.robot_emotions.metadata import get_protocol_info


def _load_real_manifest_index(real_output_root: Path) -> pd.DataFrame:
    """Carrega o manifesto real e indexa por clip_id para lookup de metadados."""
    path = real_output_root / 'virtual_imu_manifest.jsonl'
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        clip_dir = str(Path(e['artifacts']['virtual_imu_npz_path']).parent.parent.parent.resolve())
        rows.append({
            'clip_id':      str(e['clip_id']),
            'domain':       str(e['domain']),
            'user_id':      int(e['user_id']),
            'tag_number':   int(e['tag_number']),
            'take_id':      e.get('take_id'),
            'clip_dir':     clip_dir,
        })
    return pd.DataFrame(rows).set_index('clip_id')


def _load_anchor_catalog_index(anchor_catalog_path: Path) -> dict[str, tuple[float, float]]:
    """Indexa o anchor catalog por window_id → (start_sec, end_sec)."""
    index = {}
    for line in anchor_catalog_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        wid = e.get('window_id') or e.get('prompt_id')
        if wid and 'window' in e and e['window']:
            index[wid] = (float(e['window']['start_sec']), float(e['window']['end_sec']))
    return index


def _build_synthetic_capture_table(
    manifest_path: Path,
    real_index: pd.DataFrame,
    anchor_index: dict[str, tuple[float, float]],
    *,
    pose_kind_filter: str | None = None,
) -> pd.DataFrame:
    """
    Constrói capture table a partir de um manifesto sintético ou misto.

    emotion/modality/stimulus vêm de get_protocol_info(domain, tag_number) —
    as mesmas classes canônicas usadas pelo experimento real — e não dos
    labels livres do manifesto de geração.
    """
    rows = []
    for line in manifest_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        if e.get('status') not in ('ok', 'warning'):
            continue
        pose_kind = e.get('pose_kind', 'synthetic')
        if pose_kind_filter is not None and pose_kind != pose_kind_filter:
            continue

        if pose_kind == 'real':
            artifacts = e.get('artifacts', {})
            pose3d_npz = artifacts.get('pose3d_npz_path')
            virtual_imu_npz = artifacts.get('virtual_imu_npz_path')
            ref_clip_id = str(e['clip_id'])
        else:
            va = e.get('virtual_imu_artifacts', {})
            pose3d_npz = va.get('pose3d_npz_path')
            virtual_imu_npz = va.get('virtual_imu_npz_path')
            ref_clip_id = str(e.get('reference_clip_id') or e.get('clip_id'))

        if pose3d_npz is None or virtual_imu_npz is None:
            continue
        if not Path(pose3d_npz).exists() or not Path(virtual_imu_npz).exists():
            continue
        if ref_clip_id not in real_index.index:
            continue

        real_row = real_index.loc[ref_clip_id]
        protocol = get_protocol_info(real_row['domain'], real_row['tag_number']) or {}
        emotion  = str(protocol.get('emotion', ''))
        modality = str(protocol.get('modality', ''))
        stimulus = str(protocol.get('stimulus', 'None'))

        window_id = e.get('window_id')
        real_imu_time_range_sec = anchor_index.get(window_id) if window_id else None

        sample_id = str(e.get('sample_id') or e.get('prompt_id') or e['clip_id'])
        rows.append({
            'clip_id':                            sample_id,
            'reference_clip_id':                  ref_clip_id,
            'domain':                             real_row['domain'],
            'user_id':                            real_row['user_id'],
            'tag_number':                         real_row['tag_number'],
            'take_id':                            real_row['take_id'],
            'emotion':                            emotion,
            'modality':                           modality,
            'stimulus':                           stimulus,
            'status':                             str(e.get('status', 'ok')),
            'pose3d_npz_path':                    pose3d_npz,
            'virtual_imu_npz_path':               virtual_imu_npz,
            'virtual_imu_frame_aligned_npz_path': None,
            'clip_dir':                           real_row['clip_dir'],
            'pose_kind':                          pose_kind,
            'quality_report':                     {},
            'real_imu_time_range_sec':            real_imu_time_range_sec,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['subject_group'] = df.apply(
        lambda r: f"{r['domain']}_user_{int(r['user_id']):02d}", axis=1
    )
    df['flat_tag'] = df.apply(
        lambda r: f"{r['emotion']}|{r['modality']}|{r['stimulus']}", axis=1
    )
    df['frame_aligned_available'] = False
    n_with_range = df['real_imu_time_range_sec'].notna().sum()
    print(f'  real_imu_time_range_sec preenchido: {n_with_range}/{len(df)} amostras')
    return df.reset_index(drop=True)


def _apply_exclusions(df: pd.DataFrame, excluded_emotions: set[str], excluded_stimuli: set[str]) -> pd.DataFrame:
    """Remove capturas cujas emoções ou estímulos estão nas listas de exclusão."""
    if df.empty:
        return df
    mask = pd.Series(False, index=df.index)
    if excluded_emotions:
        emotion_mask = df['emotion'].isin(excluded_emotions)
        if emotion_mask.any():
            print(f'  Excluindo {int(emotion_mask.sum())} capturas com emoção em {excluded_emotions}')
        mask |= emotion_mask
    if excluded_stimuli:
        stimulus_mask = df['stimulus'].isin(excluded_stimuli)
        if stimulus_mask.any():
            print(f'  Excluindo {int(stimulus_mask.sum())} capturas com estímulo em {excluded_stimuli}')
        mask |= stimulus_mask
    return df.loc[~mask].reset_index(drop=True)


def _sort_suite_summary(summary_df: pd.DataFrame) -> pd.DataFrame:
    order_map = {name: idx for idx, name in enumerate(REQUESTED_ORDER)}
    df = summary_df.copy()
    df['_sort_key'] = df['experiment_name'].astype(str).map(order_map).fillna(len(REQUESTED_ORDER))
    return df.sort_values('_sort_key', kind='stable').drop(columns=['_sort_key']).reset_index(drop=True)


def _run_pose_experiment(
    label: str,
    captures_df: pd.DataFrame,
) -> dict:
    """Constrói dataset e roda o suite de classificadores para uma capture table."""
    captures_df = _apply_exclusions(captures_df, EXCLUDED_EMOTIONS, EXCLUDED_STIMULI)

    display(Markdown(f'### {label} — dataset'))
    display(captures_df[['clip_id', 'domain', 'user_id', 'tag_number', 'emotion', 'modality', 'stimulus', 'status']].head())
    print({'num_captures': len(captures_df), 'domains': sorted(captures_df['domain'].unique().tolist())})

    dataset_bundle = build_windowed_multimodal_dataset(
        REAL_OUTPUT_ROOT,
        config=DATASET_CONFIG,
        captures_df=captures_df,
    )
    display(dataset_bundle['metadata'].head())
    display(dataset_bundle['alignment_summary'].head())
    print({
        'num_samples':          len(dataset_bundle['metadata']),
        'pose_shape':           dataset_bundle['pose_windows'].shape,
        'imu_real_shape':       dataset_bundle['imu_real_windows'].shape,
        'imu_synthetic_shape':  dataset_bundle['imu_synthetic_windows'].shape,
    })

    if not TORCH_READY:
        display(Markdown(f'**PyTorch não disponível**: `{TORCH_IMPORT_ERROR}`'))
        return {'label': label, 'dataset_bundle': dataset_bundle, 'suite_result': None}

    display(Markdown(f'### {label} — suite de classificadores'))
    suite_result = run_experiment_suite(
        dataset_bundle,
        experiment_names=list(EXPERIMENT_SPECS.keys()),
        split_config=SPLIT_CONFIG,
        model_config=MODEL_CONFIG,
        training_config=TRAINING_CONFIG,
    )
    display(_sort_suite_summary(suite_result['summary']))
    display(suite_result['domain_gap_summary'])
    return {'label': label, 'dataset_bundle': dataset_bundle, 'suite_result': suite_result}


REAL_INDEX = _load_real_manifest_index(REAL_OUTPUT_ROOT)
ANCHOR_INDEX = _load_anchor_catalog_index(ANCHOR_CATALOG)
print(f'Real manifest index: {len(REAL_INDEX)} clips')
print(f'Anchor catalog index: {len(ANCHOR_INDEX)} janelas')

Real manifest index: 89 clips
Anchor catalog index: 3726 janelas


---
## Experimento 1 — Apenas pose real

Idêntico ao notebook original. Usa `build_classifier_capture_table` com o manifesto real.

In [ ]:
CAPTURES_REAL = build_classifier_capture_table(REAL_OUTPUT_ROOT)
RESULT_REAL   = _run_pose_experiment('real_pose', CAPTURES_REAL)

  Excluindo 4 capturas com emoção em {'Anger', 'Fear'}
  Excluindo 10 capturas com estímulo em {'Simulation'}


### real_pose — dataset

,clip_id,domain,user_id,tag_number,emotion,modality,stimulus,status
0,robot_emotions_10ms_u02_tag01,10ms,2,1,Neutrality,Standing,None,warning
1,robot_emotions_10ms_u02_tag05,10ms,2,5,Sadness,Sitting,Visual methods,warning
2,robot_emotions_10ms_u02_tag06,10ms,2,6,Sadness,Sitting,Autobiographical recall,warning
3,robot_emotions_10ms_u02_tag07,10ms,2,7,Sadness,Standing,Autobiographical recall,warning
4,robot_emotions_10ms_u02_tag09,10ms,2,9,Happiness,Sitting,Visual methods,warning


{'num_captures': 69, 'domains': ['10ms', '30ms']}


,sample_id,capture_id,clip_id,domain,user_id,tag_number,take_id,emotion,modality,stimulus,flat_tag,subject_group,window_index,window_start_index,window_size,window_overlap,quality_status,synthetic_variant,imu_feature_mode,selected_sensors,pose_imu_lag_samples,pose_imu_lag_seconds,pose_imu_correlation_before_dtw,pose_imu_correlation_after_dtw,pose_imu_dtw_normalized_distance,visible_joint_ratio,mean_confidence,temporal_jitter_score,root_drift_score,emotion_id,modality_id,stimulus_id,flat_tag_id
0,robot_emotions_10ms_u02_tag01::window_0000,robot_emotions_10ms_u02_tag01,robot_emotions_10ms_u02_tag01,10ms,2,1,1,Neutrality,Standing,None,Neutrality|Standing|None,10ms_user_02,0,0,81,0.5,warning,raw,acc_euler,"(waist, head, left_forearm, right_forearm)",-4,-0.133484,0.026847,0.671229,0.178128,0.999972,0.906851,0.000654,0.000231,2,1,1,6
1,robot_emotions_10ms_u02_tag01::window_0001,robot_emotions_10ms_u02_tag01,robot_emotions_10ms_u02_tag01,10ms,2,1,1,Neutrality,Standing,None,Neutrality|Standing|None,10ms_user_02,1,40,81,0.5,warning,raw,acc_euler,"(waist, head, left_forearm, right_forearm)",-4,-0.133484,0.026847,0.671229,0.178128,0.999972,0.906851,0.000654,0.000231,2,1,1,6
2,robot_emotions_10ms_u02_tag01::window_0002,robot_emotions_10ms_u02_tag01,robot_emotions_10ms_u02_tag01,10ms,2,1,1,Neutrality,Standing,None,Neutrality|Standing|None,10ms_user_02,2,80,81,0.5,warning,raw,acc_euler,"(waist, head, left_forearm, right_forearm)",-4,-0.133484,0.026847,0.671229,0.178128,0.999972,0.906851,0.000654,0.000231,2,1,1,6
3,robot_emotions_10ms_u02_tag01::window_0003,robot_emotions_10ms_u02_tag01,robot_emotions_10ms_u02_tag01,10ms,2,1,1,Neutrality,Standing,None,Neutrality|Standing|None,10ms_user_02,3,120,81,0.5,warning,raw,acc_euler,"(waist, head, left_forearm, right_forearm)",-4,-0.133484,0.026847,0.671229,0.178128,0.999972,0.906851,0.000654,0.000231,2,1,1,6
4,robot_emotions_10ms_u02_tag01::window_0004,robot_emotions_10ms_u02_tag01,robot_emotions_10ms_u02_tag01,10ms,2,1,1,Neutrality,Standing,None,Neutrality|Standing|None,10ms_user_02,4,160,81,0.5,warning,raw,acc_euler,"(waist, head, left_forearm, right_forearm)",-4,-0.133484,0.026847,0.671229,0.178128,0.999972,0.906851,0.000654,0.000231,2,1,1,6


...

vision_imu_s2r | fold 5/5 | Epoch 1/10:   0%|          | 0/35 [00:00<?, ?batch/s]

vision_imu_s2r | fold 5/5 | Epoch 2/10:   0%|          | 0/35 [00:00<?, ?batch/s]

vision_imu_s2r | fold 5/5 | Epoch 3/10:   0%|          | 0/35 [00:00<?, ?batch/s]

vision_imu_s2r | fold 5/5 | Epoch 4/10:   0%|          | 0/35 [00:00<?, ?batch/s]

vision_imu_s2r | fold 5/5 | Epoch 5/10:   0%|          | 0/35 [00:00<?, ?batch/s]

---
## Experimento 2 — Apenas pose sintética (Kimodo)

Cada sample é uma janela de 5 s gerada pelo Kimodo a partir de uma janela do vídeo real.
O IMU real continua vindo do clipe de referência (necessário para alinhamento e rótulos).

In [ ]:
CAPTURES_SYNTHETIC = _build_synthetic_capture_table(
    SYNTHETIC_MANIFEST, REAL_INDEX, ANCHOR_INDEX, pose_kind_filter='synthetic'
)
RESULT_SYNTHETIC   = _run_pose_experiment('synthetic_pose', CAPTURES_SYNTHETIC)

---
## Experimento 3 — Pose real + sintética (misto)

Concatena os dois conjuntos acima. Cada entrada mantém o campo `pose_kind` para rastreabilidade.

In [ ]:
CAPTURES_MIXED = _build_synthetic_capture_table(
    MIXED_MANIFEST, REAL_INDEX, ANCHOR_INDEX, pose_kind_filter=None
)
RESULT_MIXED   = _run_pose_experiment('mixed_pose', CAPTURES_MIXED)

---
## Comparação dos três experimentos

In [ ]:
frames = []
for result in (RESULT_REAL, RESULT_SYNTHETIC, RESULT_MIXED):
    if result['suite_result'] is None:
        continue
    df = _sort_suite_summary(result['suite_result']['summary']).copy()
    df.insert(0, 'pose_experiment', result['label'])
    frames.append(df)
if frames:
    display(pd.concat(frames, ignore_index=True))

## Confusion matrices (melhor fold por experimento)

In [ ]:
if TORCH_READY:
    from evaluation.classifiers.metrics import plot_confusion_matrices

    for result in (RESULT_REAL, RESULT_SYNTHETIC, RESULT_MIXED):
        suite = result['suite_result']
        if suite is None:
            continue
        best_experiment_name = suite['summary'].iloc[0]['experiment_name']
        best_result = max(
            (r for r in suite['results'] if r['experiment_name'] == best_experiment_name),
            key=lambda r: float(r['metrics'].get('global_score_macro_f1_mean') or float('-inf')),
        )
        split_suffix = '' if best_result.get('split_id') is None else f" (split {best_result['split_id']})"
        fig, axes = plot_confusion_matrices(best_result['metrics'], result['dataset_bundle']['label_encoders'])
        fig.suptitle(f"{result['label']} — {best_experiment_name}{split_suffix}", y=1.02)
        plt.show()